# TSV Concentric · HFSS Eigenmode

Use the same Concentric Signal/stitched Ground system as the [Q3D
notebook](aedt_q3d.ipynb), now in an HFSS Eigenmode design. The default
full-run flow searches three modes above 1 GHz, with at most three
adaptive passes and 1% delta-frequency convergence. These
Human-confirmed first-trial controls are not an accuracy or
scientific-success criterion. This example has no junction, lumped load,
drive port or EPR request; it is not a qubit-frequency calculation.
Native execution and shared-runtime admission remain pending; this
source authoring is not a solved result. See the [TSV
guide](../../../../simulation/tsv-model-inspection.qmd) and [Eigenmode
requirement](../../../../simulation/requirements/eigenmode.qmd).

Fresh construction uses the local shared-physical candidate, not a
published wheel. Retained results keep their original runtime
identities.

Published historical baseline:
[SCGSim](https://github.com/OrPenStrike/scgsim/tree/c9270c7e76e0b2df187dddde462079e00d410dd0).

## Setup and Imports

Public source and model-record imports do not open an AEDT Desktop.

## Setup and Run Controls

In [ ]:
import hashlib
import subprocess
import sys
from dataclasses import replace
from importlib.metadata import version
from pathlib import Path
from uuid import uuid4

from IPython.display import display
from scgsim.aedt import (
    EigenmodeRunControl,
    prepare_hfss_eigenmode_from_geometry,
    prepare_planar_geometry_input,
    resolve_prepared_model,
    resolve_results,
)
from scgsim.geometry import (
    ApproximationProfile,
    GeometryBuildInput,
    LayoutPolygonSpec,
    SemanticEntitySpec,
)

import orcapen_sc_pdk
from orcapen_sc_pdk.cells import get_tsv_source_records, tsv_concentric_signal_coupon

# Actions: run | prepare_handoff | analyze_handoff | source | build_model | inspect_model
WORKFLOW_ACTION = "run"
DIE = "D0"
DIE_SIDE_UM = 1000.0
BORE_DIAMETER_UM = 60.0
WALL_THICKNESS_UM = 0.2
SIGNAL_OUTER_RADIUS_UM = 100.0
GAP_UM = 20.0
GROUND_TSV_OFFSET_UM = 300.0
MODEL = "thin_film"
APPROXIMATION_PROFILE = ApproximationProfile(name=MODEL, revision="v1", parameters={})
SETUP_NAME = "Setup1"
MINIMUM_FREQUENCY_GHZ = 1.0
NUM_MODES = 3
MAXIMUM_PASSES = 3
MAXIMUM_DELTA_FREQUENCY_PERCENT = 1.0
MINIMUM_PASSES = 1
MINIMUM_CONVERGED_PASSES = 1
PERCENT_REFINEMENT = 30.0
RUN_CONTROL = (
    None
    if WORKFLOW_ACTION == "build_model"
    else EigenmodeRunControl(
        setup_name=SETUP_NAME,
        minimum_frequency_ghz=MINIMUM_FREQUENCY_GHZ,
        num_modes=NUM_MODES,
        maximum_passes=MAXIMUM_PASSES,
        maximum_delta_frequency_percent=MAXIMUM_DELTA_FREQUENCY_PERCENT,
        minimum_passes=MINIMUM_PASSES,
        minimum_converged_passes=MINIMUM_CONVERGED_PASSES,
        percent_refinement=PERCENT_REFINEMENT,
    )
)
RUN_ID = f"tsv-concentric-eigenmode-{uuid4().hex[:12]}"
OUTPUT_ROOT = Path(".artifacts/SCGSimExamples/TSV")
RUN_DIR = OUTPUT_ROOT / RUN_ID
SOURCE_DIR = OUTPUT_ROOT / f"{RUN_ID}-source"
RETURNED_RUN_DIR = None  # Set an explicit actual root for analyze_handoff or inspect_model.
PROJECT_NAME = "TSVConcentricEigenmodeThinFilm"
DESIGN_NAME = "Eigenmode"
REGION_PADDING_XYZ_UM = (250.0, 250.0, 1000.0)
GEOMETRY_WORKERS = 1
RESOURCES = None  # No host allocation selected here; native admission is separate.
SIGNAL_ENTITY_IDS = (f"{DIE}/TOP_SIGNAL", f"{DIE}/BOTTOM_SIGNAL", f"{DIE}/SIGNAL_WALL")
GROUND_ENTITY_IDS = (f"{DIE}/TOP_GROUND", f"{DIE}/BACK_GROUND") + tuple(
    f"{DIE}/GROUND_TSV_{i}" for i in range(4)
)
if WORKFLOW_ACTION not in {
    "run",
    "prepare_handoff",
    "analyze_handoff",
    "source",
    "build_model",
    "inspect_model",
}:
    raise ValueError("Unknown WORKFLOW_ACTION")
display({name: version(name) for name in ("orcapen-sc-pdk", "gdsfactory", "scgsim")})

`run` prepares a fresh handoff, invokes the full-solve CLI and reads
results. `prepare_handoff` stops at portable inputs; `source` stays at
layout/source facts. `analyze_handoff` reads an explicit returned root
without a factory, getter, GDS write, preparation or CLI call.
Separately, `build_model` passes `run_control=None` to the candidate’s
model-only Prepare Only transaction; `inspect_model` reads that returned
model. Neither is a Model+Setup/reuse API. Resource flags are not a
CAD/aggregate hard cap; native admission is separate.

## Create Simulation Component / Coupon

In [ ]:
if WORKFLOW_ACTION not in {"analyze_handoff", "inspect_model"}:
    orcapen_sc_pdk.activate()
    coupon = tsv_concentric_signal_coupon(
        die=DIE,
        die_side_um=DIE_SIDE_UM,
        bore_diameter_um=BORE_DIAMETER_UM,
        wall_thickness_um=WALL_THICKNESS_UM,
        signal_outer_radius_um=SIGNAL_OUTER_RADIUS_UM,
        gap_um=GAP_UM,
        ground_tsv_offset_um=GROUND_TSV_OFFSET_UM,
    )
    coupon.plot()

The central hollow wall joins the two annular Signal pads by source
intent; four outer hollow walls join the Ground planes. Silicon is
explicitly cut at all five bores. M1 and substrate thickness come from
the selected PDK levels.

## Initialize AEDT Project / App

The later `run` or `build_model` CLI initializes its own HFSS Eigenmode
Desktop/design. Source authoring below does not attach to an existing
Human or solver session.

## Import GDS and Build the HFSS/Q3D/Q2D Model

The canonical OrcaPen getter supplies actual placed polygon exteriors,
holes, source occurrences, physical levels and materials. This cell only
adopts the explicit case Net grouping and constructs SCGSim’s public
input records. It does not infer connectivity or register invented
children. This cell constructs source input; actual native import/build
belongs to the later CLI transaction, not this source observation.

In [ ]:
if WORKFLOW_ACTION not in {"analyze_handoff", "inspect_model"}:
    bounds = coupon.dbbox()
    regions = coupon.info["tsv_source"]["source_regions"]
    source_bottom = min(r["z_interval_um"][0] for r in regions)
    source_top = max(r["z_interval_um"][1] for r in regions)
    px, py, pz = REGION_PADDING_XYZ_UM
    background = {
        "semantic_id": "CALLER_BACKGROUND",
        "role": "solution_region",
        "material_id": "vacuum",
        "material_kind": "vacuum",
        "priority": 99,
        "geometry_kind": "domain",
        "geometry": {
            "domain_bounds_um": {
                "x_min_um": bounds.left - px,
                "x_max_um": bounds.right + px,
                "y_min_um": bounds.bottom - py,
                "y_max_um": bounds.top + py,
            },
            "z_min_um": source_bottom - pz,
            "z_max_um": source_top + pz,
        },
        "metadata": {"meaning": "computational background complement"},
    }
    source = get_tsv_source_records(coupon, host_solution_region=background)
    caller_nets = {
        **dict.fromkeys(SIGNAL_ENTITY_IDS, "Signal"),
        **dict.fromkeys(GROUND_ENTITY_IDS, "Ground"),
    }
    records = source["geometry_input"]
    stack = source["stack"]
    for layer in stack["layers"]:
        layer["net_id"] = caller_nets[layer["semantic_id"]]
    # Keep physical material records; add only the compiler's kind vocabulary.
    geometry_kinds = {
        "superconductor": "conductor",
        "dielectric": "dielectric",
        "vacuum": "vacuum",
    }
    stack["materials"] = {
        name: {
            **record,
            "kind": geometry_kinds[record["material_kind"]],
            "permittivity": record["relative_permittivity"],
        }
        for name, record in stack["materials"].items()
    }
    build_input = GeometryBuildInput(
        polygons=tuple(LayoutPolygonSpec(**p) for p in records["polygons"]),
        entities=tuple(
            SemanticEntitySpec(**{**e, "net_id": caller_nets.get(e["semantic_id"])})
            for e in records["entities"]
        ),
        solution_regions=records["solution_regions"],
        metadata={**records["metadata"], "caller_final_nets": caller_nets},
    )
    display(
        {
            "source_parameters": coupon.info["tsv_source"]["parameters"],
            "source_contacts": records["metadata"]["source_contacts"],
            "source_ports": records["metadata"]["source_ports"],
            "caller_nets": caller_nets,
            "physical_layer_declarations": {
                e.semantic_id: e.metadata["physical_layer"]
                for e in build_input.entities
                if e.material_kind == "conductor"
            },
        }
    )

## Geometry Verification

These are source observations. Silicon already has the authored bore
holes; the TSV wall is an annulus, not a filled cylinder. In
`thin_film`, face films follow their declared contact side while TSV
`retained_solid` levels stay finite. SCGSim owns the effective-Z mapping
and the actual backend geometry. The current AEDT path consumes sampled
GDS polygon witnesses; exact circular recipes remain source intent, not
active native curve reconstruction.

### Fresh source witness

In [ ]:
if WORKFLOW_ACTION in {"run", "prepare_handoff", "build_model"}:
    SOURCE_DIR.mkdir(parents=True, exist_ok=False)
    gds_path = coupon.write_gds(SOURCE_DIR / "source.gds", with_metadata=False)
    gds_bytes = Path(gds_path).read_bytes()
    gds_sha256 = hashlib.sha256(gds_bytes).hexdigest()
    stack["metadata"]["gds_sha256"] = gds_sha256
    build_input = replace(build_input, metadata={**build_input.metadata, "gds_sha256": gds_sha256})

The exported witness and normalized source records come from the same
component. The backend helper owns actual lowering; this notebook does
not interpret a source GDS hash as an actual native geometry
observation.

## Materials and Boundaries

Al and TiN remain the physical source materials; their PDK
superconducting declarations select ideal native conductor treatment.
Silicon is a perforated dielectric solid. The caller-authored background
is the surrounding complement, not a second overlapping occupied body.
Final Nets are the explicit grouping above, not a connectivity result
inferred by the notebook.

## Simulation Setup

The notebook’s explicit run control selects the Human-confirmed
first-trial physics settings before preparation. `build_model` alone
passes `run_control=None` for the candidate’s model-only transaction,
without an Analysis Setup or sweep. The getter now carries the same
selected neutral `physical_layer` declaration on Entity and stack
records. Shared interpretation belongs to SCGSim, not notebook
arithmetic. Read-only returned-run actions skip every
component/getter/GDS/preparation stage above.

## Simulation Configuration

In [ ]:
HANDOFF = None
if WORKFLOW_ACTION in {"run", "prepare_handoff", "build_model"}:
    geometry = prepare_planar_geometry_input(
        build_input, prepared_stack=stack, modeling=APPROXIMATION_PROFILE.name
    )
    HANDOFF = prepare_hfss_eigenmode_from_geometry(
        modeling=APPROXIMATION_PROFILE.name,
        geometry=geometry,
        project_name=PROJECT_NAME,
        design_name=DESIGN_NAME,
        run_control=RUN_CONTROL,
        output_dir=RUN_DIR,
        geometry_workers=GEOMETRY_WORKERS,
        resources=RESOURCES,
    )
    display(HANDOFF)

The surrounding Region follows the explicit enclosure. Closed HFSS
exterior treatment is retained: no Radiation or PML is requested. The
run control selects the explicit search/pass settings; only
`build_model` omits the Analysis Setup by passing `run_control=None`. No
junction, load or EPR request is added. Thin Film affects face metal
only; vertical TSV walls retain their finite volume.

## Solve and Export

`run` invokes the actual full-solve CLI. `build_model` alone adds the
real `--prepare-only` flag for build/readback/save/release without
Analyze. Native failures propagate through `check=True` without retry or
fabricated results.

In [ ]:
if WORKFLOW_ACTION in {"run", "build_model"}:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "scgsim.aedt.run",
            "--handoff",
            str(HANDOFF.metadata_path),
            "--execute",
        ]
        + (["--prepare-only"] if WORKFLOW_ACTION == "build_model" else []),
        check=True,
    )

## Adaptive-Pass Convergence / Solver Diagnostics

Read actual returned numerical/performance evidence before interpreting
modes. Three passes is a short first-trial endpoint, not convergence
proof. Missing or corrupt results and save/release failures remain
strict resolver errors.

In [ ]:
RESULT = None
PREPARED_MODEL = None
if WORKFLOW_ACTION in {"run", "analyze_handoff"}:
    RESULT = resolve_results(HANDOFF.run_dir if WORKFLOW_ACTION == "run" else RETURNED_RUN_DIR)
    RESULT.show_all_results(show_details=True)
elif WORKFLOW_ACTION in {"build_model", "inspect_model"}:
    PREPARED_MODEL = resolve_prepared_model(
        HANDOFF.run_dir if WORKFLOW_ACTION == "build_model" else RETURNED_RUN_DIR
    )

## Results: Plots and Readable Tables

### Physics Analysis Results

Display actual mode/frequency results with their ordering, units and
returned artifact identity; do not infer physical mode labels from row
position. No EPR or junction calculation is requested. Model-only
readback instead covers sheets, solids, effective Z, bores and
materials; native query missingness does not prove electrical continuity
or modal physics.

In [ ]:
if RESULT is not None:
    display(RESULT.physics_results())
elif PREPARED_MODEL is not None:
    display(PREPARED_MODEL.project_path)
    display(PREPARED_MODEL.readback)

### Simulation Performance / Benchmarks

Display actual solved timing/pass/mesh evidence separately from physics.
The model-only branch retains preparation/save/release evidence without
solved benchmarks or adaptive histories.

In [ ]:
if RESULT is not None:
    display(RESULT.simulation_benchmark())
elif PREPARED_MODEL is not None:
    display({"mode": PREPARED_MODEL.mode, "model_stage": PREPARED_MODEL.model_stage})

## Save and Release AEDT

The native transaction saves and releases its own Desktop. Open a
separate writable project copy with its necessary companions for Human
geometry review; do not edit the hash-bound producer. Earlier Solid
projects retain their own execution identity and are not current
notebook outputs.